# A dispatch model with capacity expansion

This script introduces you to the code of an electricity dispatch model with capacity expansion. You are able to execute this document and receive results.

For a general introduction to the dispatch model, please consult the previous chapter *A simple dispatch model*. This model extends the dispatch model by the ability to expand the energy system under certain constraints.

The following code is the implementation of the model. For a simple example case with limited data and little calculation effort, the script is sufficient. 

Note: the representation of the investment cost is not comparable to a financial analysis. It rather gives an indication how the system will look like under specific cost characteristics. Further, this example establishes a greenfield approach.

We start the setup by loading the relevant packages with the 'using' command. Packages are a collection of pre-defined functions and tools that are helpful for our programming efforts.

In [ ]:
using JuMP
using Clp
using Plots
using DataFrames, CSV

## Loading Data

Before we implement the model, we need to define the information that is needed to build the model. We start with the data load from two CSV files. The first one with the name *timedata.csv* contains timeseries for demand and availability. The second one with the name *technologies.csv* includes cost data and characteristics of the technologies.

Note: this is an example and the data can be exchanges with other data on energy systems as well as expanded.

In [ ]:
data_path = "data" #sets the path to where the data is saved.
time_series = CSV.read(joinpath(data_path, "timedata.csv"),DataFrame)
tech_data = CSV.read(joinpath(data_path, "technologies.csv"),DataFrame)

Based on the files we define the different sets that are needed for the model. In our example, we have timesteps *t*, the power plants *p*, of which some belong to the dispatchable technologies *disp* and some to the non-dispatchable ones *nondisp*. Storage units are part of the set *s*.

The timesteps are collected from the timeseries data and are equal to the size (rows) of the given data set. The technology set originates from the technology data and collects all rows of the technology column into a vector (brackets are read as [row,column], *:* indicates to select all elements). The distinction between dispatchable and non-dispatchable technology is made based on the second column in the file *dispatchable*. A *1* is assigned if true, a *0* if false. The elements from the rows are therefore collected based on a condition from column *technology*. For storage *s*, we select the rows in the *technology* column that have a value greater than zero in the column *investment_storage*. This is an indicator for a storage technology.

In [ ]:
T = 1:size(time_series, 1) |> collect
P = tech_data[:,:technology] |> Vector
DISP = tech_data[tech_data[!,:dispatchable] .== 1, :technology]
NONDISP = tech_data[tech_data[!,:dispatchable] .== 0 ,:technology]
S = tech_data[tech_data[!,:investment_storage] .> 0 ,:technology]

Demand and production from renewable energy sources vary over time and enter the model as exogenous information. The data is read from the *time_series* data frame from the column *demand* and the columns for the non-dispatchable technologies. The demand is saved as a vector and the availability as a Dictionary where each technology (wind and pv in our model) have their own timeseries.

In [ ]:
demand = time_series[:,:demand] |> Vector
availability = Dict(nondisp => time_series[:,nondisp] for nondisp in NONDISP)

For the capacity extension, there is a need for financial data that goes beyond capacity restrictions, demand data and availability. As a basis for investment cost, we use overnight cost and calculate an annuity of them.

\begin{align*}
annuity_p &= \frac{OC_p \cdot r}{1- e^{-T_p \cdot r}} \approx OC_p \cdot \frac{1}{1-(1+r)^{-T}} = OC_p \cdot \text{annuity\_factor}
\end{align*}

For that, we need the annuity factor as: 

In [ ]:
# Implementation of annuity factor equation
annuity_factor(n,r) = r * (1+r)^n / (((1+r)^n)-1)

Next, we define the interest rate *r* and initialise the dictionaries to store the metrics for the technologies. We assign cost for investing into generation capacity, charging capacity, storage capacity and variable cost for the different technologies. Further, we save efficiency data for the storage.

In [ ]:
# Set the interest rate for the annuity calculations.
interest_rate = 0.04

# Initialize dictionaries to store various metrics for each technology.
# The keys are strings (names of technologies), and the values are floating-point numbers.
ic_generation_cap = Dict{String, Float64}()
ic_charging_cap = Dict{String, Float64}()
ic_storage_cap = Dict{String, Float64}()
vc = Dict{String, Float64}()

eff_in = Dict{String, Float64}()
eff_out = Dict{String, Float64}()

To fill the initiated dictionaries, we implement a loop that reads data from our *tech_data*. The loop iterates over each row of the DataFrame and calculates the specific annuity factor based on the individual lifetime, it calculates and stores the cost for the generation capacity based on the total investment cost. For the charge, storage, and the efficiencies, the *&&* operator is used to perform a logical ‘AND’ operation, ensuring that only positive investment costs and efficiencies are stored in the dictionaries. Last, all variable cost are saved for the technologies.

In [ ]:
# Iterate over each row of the `tech_data` dataset.
for row in eachrow(tech_data)
    # Calculate the annuity factor based on the technology's lifetime and the interest rate.
    af = annuity_factor(row.lifetime, interest_rate)

    # Calculate and store the investment cost for generation capacity, adjusted by the annuity factor.
    ic_generation_cap[row.technology] = row.investment_generation * af

    # Calculate the investment cost for charging capacity, adjusted by the annuity factor.
    # Store it if it's greater than 0.
    iccc = row.investment_charge * af
    iccc > 0 && (ic_charging_cap[row.technology] = iccc)

    # Calculate the investment cost for storage capacity, adjusted by the annuity factor.
    # Store it if it's greater than 0.
    icsc = row.investment_storage * af
    icsc > 0 && (ic_storage_cap[row.technology] = icsc)

    # Store the storage efficiency for input and output if they are greater than 0.
    row.storage_efficiency_in > 0 && (eff_in[row.technology] = row.storage_efficiency_in)
    row.storage_efficiency_out > 0 && (eff_out[row.technology] = row.storage_efficiency_out)

    # Store the variable cost for each technology.
    vc[row.technology] = row.vc
end

For the storage we need a function that checks for the next period (successor), and to scale the cost to the dispatch, we divide the hours of the year by the length of the chosen timeseries.

In [ ]:
successor(arr, x) = (x == length(arr)) ? 1 : x + 1
dispatch_scale = 8760/length(T)

### The model

The model of a dispatch with capacity expansion comprises the following equations:

$$
\begin{align*}
    min~C^{gen} + C^{inv} &= \sum_{p,t} mc_{disp} \cdot G_{disp,t} \cdot scale^{ty}
    + \sum_{p}  annuity_p \cdot CAP^{G/D/L}_{p}\\
\end{align*}
$$

$$
\begin{align*}
    s.t. \qquad & d_{t} = \sum_p G_{disp,t} + feed\_in_{ndisp,t} - CU_t - D^{stor}_{s,t} \qquad &\forall t \in T\\
    &G_{disp,t}  \leq CAP_{disp} \qquad &\forall disp \in DISP, t \in T\\
    &feed\_in_{ndisp} = ava_{ndisp,t} \cdot CAP^G_{ndisp} \qquad &\forall ndisp \in NONDISP, t \in T\\
    &L^{stor}_{s,t+1} = L^{stor}_{s, t} - G_{s,t} + D^{stor}_{s,t} &\forall s \in S, t \in T\\
    &G_{s,t} = CAP^G_s  &\forall s \in S\\
    &D^{stor}_{s,t}  \leq CAP^D_{s}  &\forall s \in S, t \in T\\
    &L^{stor}_{s,t} \leq CAP^L_{s}  &\forall s \in S, t \in T\\
    &0 \leq G_{p,t},~CAP^G_p, ~CAP^D_s,~CAP^L_s, CU_t
\end{align*}
$$

At first, we initialise the model and define all variables. While generation $G_{disp, t}$, curtailment $CU_t$, storage discharge $D^{stor}_{s,t}$, and storage level $L^{stor}_{s,t}$ have been part of all previous dispatch models, we introduce three new variables here that represent the capacity expansion. Generation expansion $CAP^G_{p}$, discharge expansion $CAP^D_{s}$ and storage expansion $CAP^L_{s}$ are new in this model.

In [ ]:
m = Model(Clp.Optimizer)

@variables m begin
    # variables from our dispatch model
    G[DISP, T] >= 0
    CU[T] >= 0
    D_stor[S,T] >= 0
    L_stor[S,T] >= 0

    # new variables for our investment model
    CAP_G[P] >= 0
    CAP_D[S] >= 0
    CAP_L[S] >= 0
end

The objective minimises the cost. These are the variable cost for the dispatch scaled to a full year and the investment expenses for generation, storage and storage capacity expansion.

In [ ]:
@objective(m, Min,
    sum(vc[disp] * G[disp,t] for disp in DISP, t in T) * dispatch_scale
    + sum(ic_generation_cap[p] * CAP_G[p] for p in P)
    + sum(ic_charging_cap[s] * CAP_D[s] for s in S if haskey(ic_charging_cap, s))
    + sum(ic_storage_cap[s] * CAP_L[s] for s in S)
)

As a different way to express renewables, we add an equation (expression) that calculates the hourly feed in based on the availability time series and the installed capacity.

In [ ]:
@expression(
    m, feed_in[ndisp=NONDISP, t=T],
    availability[ndisp][t]*CAP_G[ndisp]
)

The demand-supply balance ensures that production from renewables, dispatchable sources and storage add to demand and consumption (charge) of storage.

In [ ]:
@constraint(m, ElectricityBalance[t=T],
    sum(G[disp,t] for disp in DISP)
    + sum(feed_in[ndisp,t] for ndisp in NONDISP)
    - sum(D_stor[s,t] for s in S)
    - CU[t]
    ==
    demand[t]
)

The capacity constraints are now established as a somewhat flexible constraint. Instead of a system constraint for generation, charge and storage level, the maximum generation is limited by the variable representing capacity expansion. This capacity is now determined through the lowest cost.

In [ ]:
@constraint(m, MaxGeneration[disp=DISP, t=T],
    G[disp,t] <= CAP_G[disp]
)

@constraint(m, MaxCharge[s=S, t=T; haskey(ic_charging_cap, s)],
    D_stor[s,t] <= CAP_D[s]
)

@constraint(m, SymmetricChargingPower[s=S, t=T; !(haskey(ic_charging_cap, s))],
    CAP_G[s] == CAP_D[s]
)

@constraint(m, MaxLevel[s=S, t=T],
    L_stor[s,t] <= CAP_L[s]
)

Last, we implement the storage level equation that call the efficiencies. We then optimise the model.

In [ ]:
@constraint(m, StorageLevel[s=S, t=T],
    L_stor[s, successor(T,t)]
    ==
    L_stor[s, t]
    + eff_in[s]*D_stor[s,t]
    - (1/eff_out[s]) * G[s,t]
)

optimize!(m)